In [ ]:
import sqlite3
import pandas as pd
from shelf_life.preprocessing import preprocess
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
conn = sqlite3.connect("../data/db/products.db")
df = pd.read_sql("SELECT * FROM product_details;", conn)

conn.close()

## Product stats

In [ ]:
df = preprocess(df)

print(df.info())
print('\nUnique skus per product code')
print(df.groupby('target_url')['sku_id'].nunique().describe())

print('\nBrand distribtion')
ax = df.brand_name.value_counts().plot(kind='hist',title='Dist of number of products per brand')
plt.show()
print(f"\nUnique brands: {df.brand_name.nunique()}")

In [ ]:
df[df["variation_type"] == "Size"].groupby(
    ["brand_name", "product_code", "display_name", "variation_value", "size_refinement"]
).agg({"sku_id": "count", "price": "first"})

In [ ]:
# how many are missing a size?
df[df['size'].isna()]
df[df['size']==""]['category_name'].value_counts()
# +300 product codes missing size information
df[df['size']==""][['display_name', 'full_product_url']].head()

# need to update preproccessing script, assume oz is floz if ml is present 

In [ ]:
df[['product_code', 'display_name', 'price','size','type','size_oz',
       'size_fl_oz', 'size_ml', 'size_g', 'size_phase', 'size_info',
       'price_per_oz', 'price_per_ml', 'price_per_g']]

In [ ]:
## l1_cat minisize might be separated from fullsize versions in other categories 
# need reliable way to link all versions of products when they may have diff product code+skus

In [ ]:
perf_df = df[df['cat_l3']=='Perfume']

In [ ]:
perf_df.groupby('product_code').agg({
    'size': 'unique',
    'type': 'unique',
    'price':'unique',
    'size_oz':'unique',
    'price_per_oz':'unique'
})

In [ ]:


plot_df = (
    perf_df
    .dropna(subset=['size_oz', 'price'])
    .drop_duplicates(subset=['product_code', 'size_oz', 'price'])
    .sort_values(['product_code', 'size_oz'])          # lines connect in size order
)
# only products with 2+ sizes are interesting as lines
plot_df = plot_df[plot_df.groupby('product_code')['size_oz'].transform('nunique') > 1]

fig = px.line(
    plot_df,
    x='size_oz', y='price',
    line_group='product_code',
    hover_name='display_name',                         # or 'product_code'
    hover_data={'brand_name': True, 'size': True, 'price_per_oz': ':.2f'},
    markers=True,
    # log_x=True, log_y=True,
    labels={'size_oz': 'Size (oz)', 'price': 'Price (CAD)'},
)
fig.update_traces(line=dict(color='rgba(100,100,100,0.25)', width=1),
                  marker=dict(size=5, color='rgba(31,119,180,0.7)'))
fig.update_layout(showlegend=False, template='simple_white')
fig.show()

In [ ]:
import numpy as np

def slope(d):
    return np.polyfit(np.log(d.size_oz), np.log(d.price), 1)[0]

exps = (plot_df.groupby('product_code')
        .apply(slope)
        .rename('b')
        .reset_index()
        .merge(plot_df[['product_code', 'display_name', 'brand_name']].drop_duplicates('product_code')))

print(exps.b.describe())

fig = px.strip(exps, x='b', hover_name='display_name', hover_data=['brand_name'],
               labels={'b': 'Price exponent (1 = linear, <1 = volume discount)'})
fig.add_vline(x=1, line_dash='dash')
fig.update_layout(template='simple_white')
fig.show()